<a href="https://colab.research.google.com/github/cpython-projects/python_da_06_11_25/blob/main/lesson_27_part_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Запити з кількох таблиць

## Легенда

### `clients` — список зареєстрованих клієнтів:

| client\_id | name  |
| ---------- | ----- |
| 1          | Anna  |
| 2          | Boris |
| 3          | Clara |
| 4          | David |

---

### `orders` — список покупок:

| order\_id | client\_id | total\_uah |
| --------- | ---------- | ---------- |
| 101       | 1          | 500        |
| 102       | 3          | 800        |
| 103       | 3          | 1200       |
| 104       | 5          | 300        |

## JOIN

JOIN — це спосіб **“зшити”** рядки з різних таблиць за спільною ознакою (найчастіше — ідентифікатором).

### Структура JOIN

```sql
SELECT ...
FROM таблиця_1
[INNER] JOIN таблиця_2
  ON таблиця_1.ключ = таблиця_2.ключ
```

* **JOIN** з’єднує рядки двох таблиць
* **ON** вказує, за якою ознакою
* Виводяться **усі рядки**, де збігся ключ

## Типи JOIN і як вони працюють

| JOIN         | Що повертає                                 |
| ------------ | ------------------------------------------- |
| `INNER JOIN` | Тільки рядки, що збігаються, з обох таблиць |
| `LEFT JOIN`  | Усі рядки з лівої таблиці + збіги з правої  |
| `RIGHT JOIN` | Усі рядки з правої таблиці + збіги з лівої  |
| `FULL JOIN`  | Усі рядки з обох таблиць, навіть без збігів |


### `INNER JOIN`

```sql
SELECT clients.client_id, clients.name, orders.total_uah
FROM clients
INNER JOIN orders
   ON clients.client_id = orders.client_id;
```

Покаже лише тих клієнтів, у кого є хоча б одне замовлення:

| client\_id | name  | total\_uah |
| ---------- | ----- | ---------- |
| 1          | Anna  | 500        |
| 3          | Clara | 800        |
| 3          | Clara | 1200       |

---

### `LEFT JOIN`

```sql
SELECT clients.client_id, clients.name, orders.total_uah
FROM clients
LEFT JOIN orders
   ON clients.client_id = orders.client_id;
```

Покаже всіх клієнтів, навіть якщо у них **немає замовлень** (у такому випадку `total_uah` буде `NULL`):

| client\_id | name  | total\_uah |
| ---------- | ----- | ---------- |
| 1          | Anna  | 500        |
| 2          | Boris | NULL       |
| 3          | Clara | 800        |
| 3          | Clara | 1200       |
| 4          | David | NULL       |

---

### `RIGHT JOIN`

```sql
SELECT clients.client_id, clients.name, orders.total_uah
FROM clients
RIGHT JOIN orders
   ON clients.client_id = orders.client_id;
```

Покаже всі замовлення, навіть якщо клієнт **не зареєстрований** (у такому випадку `name` буде `NULL`):

| client\_id | name  | total\_uah |
| ---------- | ----- | ---------- |
| 1          | Anna  | 500        |
| 3          | Clara | 800        |
| 3          | Clara | 1200       |
| 5          | NULL  | 300        |

---

### `FULL JOIN`

```sql
SELECT clients.client_id, clients.name, orders.total_uah
FROM clients
FULL JOIN orders
   ON clients.client_id = orders.client_id;
```

Покаже всіх клієнтів і всі замовлення — навіть якщо **немає відповідності** між таблицями:

| client\_id | name  | total\_uah |
| ---------- | ----- | ---------- |
| 1          | Anna  | 500        |
| 2          | Boris | NULL       |
| 3          | Clara | 800        |
| 3          | Clara | 1200       |
| 4          | David | NULL       |
| 5          | NULL  | 300        |


## JOIN на 3 таблицях: `clients`, `orders`, `order_items`

**`clients`**

| client\_id | name  |
| ---------- | ----- |
| 1          | Anna  |
| 2          | Boris |
| 3          | Clara |
| 4          | Dana  |

**`orders`**

| order\_id | client\_id | total\_uah |
| --------- | ---------- | ---------- |
| 101       | 1          | 500        |
| 102       | 3          | 800        |
| 103       | 3          | 1200       |
| 104       | NULL       | 250        |

**`order_items`**

| order\_id | product\_name | qty | price\_uah |
| --------- | ------------- | --- | ---------- |
| 101       | Зошит         | 2   | 50         |
| 101       | Ручка         | 1   | 30         |
| 102       | Пенал         | 1   | 300        |
| 999       | Гумка         | 1   | 15         |

> Тут є:
>
> * клієнт **без замовлень** (`Dana`)
> * замовлення без клієнта (`order_id = 104`)
> * замовлення, у якого **немає товарів**
> * товар, не пов’язаний із замовленням (`order_id = 999`)

### INNER JOIN (тільки збіги)

```sql
SELECT clients.name, orders.order_id, order_items.product_name
FROM clients
JOIN orders
   ON clients.client_id = orders.client_id
JOIN order_items
   ON orders.order_id = order_items.order_id;
```

**Результат:**

| name  | order\_id | product\_name |
| ----- | --------- | ------------- |
| Anna  | 101       | Зошит         |
| Anna  | 101       | Ручка         |
| Clara | 102       | Пенал         |

---

### LEFT JOIN (усі клієнти, навіть без замовлень)

```sql
SELECT clients.name, orders.order_id, order_items.product_name
FROM clients
LEFT JOIN orders
    ON clients.client_id = orders.client_id
LEFT JOIN order_items
    ON orders.order_id = order_items.order_id;
```

**Результат:**

| name  | order\_id | product\_name |
| ----- | --------- | ------------- |
| Anna  | 101       | Зошит         |
| Anna  | 101       | Ручка         |
| Boris | NULL      | NULL          |
| Clara | 102       | Пенал         |
| Clara | 103       | NULL          |
| Dana  | NULL      | NULL          |

---

### RIGHT JOIN (усі товари, навіть без клієнта)

```sql
SELECT clients.name, orders.order_id, order_items.product_name
FROM clients
RIGHT JOIN orders
    ON clients.client_id = orders.client_id
RIGHT JOIN order_items
    ON orders.order_id = order_items.order_id;
```

**Результат:**

| name  | order\_id | product\_name |
| ----- | --------- | ------------- |
| Anna  | 101       | Зошит         |
| Anna  | 101       | Ручка         |
| Clara | 102       | Пенал         |
| NULL  | 999       | Гумка         |

---

### FULL JOIN (усе звідусіль)

```sql
SELECT clients.name, orders.order_id, order_items.product_name
FROM clients
FULL JOIN orders
    ON clients.client_id = orders.client_id
FULL JOIN order_items
    ON orders.order_id = order_items.order_id;
```

**Результат:**

| name  | order\_id | product\_name |
| ----- | --------- | ------------- |
| Anna  | 101       | Зошит         |
| Anna  | 101       | Ручка         |
| Clara | 102       | Пенал         |
| Clara | 103       | NULL          |
| NULL  | 104       | NULL          |
| NULL  | 999       | Гумка         |
| Boris | NULL      | NULL          |
| Dana  | NULL      | NULL          |


## Комбінований JOIN

Потрібно:

* взяти **усіх клієнтів** (навіть без замовлень) → `LEFT JOIN`
* при цьому до замовлень під’єднати **тільки ті товари, які реально є** → `INNER JOIN`

```sql
SELECT clients.name, orders.order_id, order_items.product_name
FROM clients
LEFT JOIN orders
    ON clients.client_id = orders.client_id
INNER JOIN order_items
    ON orders.order_id = order_items.order_id;
```

---

⚠️ **Порядок JOIN-ів має значення!**

Наприклад:

* взяти **усі замовлення** (пов’язані й не пов’язані з клієнтами) → `FULL JOIN`
* але з замовлень отримати лише товари, якщо вони є → `LEFT JOIN`

```sql
SELECT orders.order_id, clients.name, order_items.product_name
FROM clients
FULL JOIN orders
    ON clients.client_id = orders.client_id
LEFT JOIN order_items
    ON orders.order_id = order_items.order_id;
```

# Агрегації, фільтрація агрегатів і вкладені запити

## `GROUP BY`: групування даних

`GROUP BY` використовується, щоб **згрупувати рядки за значенням поля**, і застосувати агрегатні функції до кожної групи:

| Функція           | Що робить           |
| ----------------- | ------------------- |
| `COUNT()`         | Рахує кількість     |
| `SUM()`           | Підсумовує значення |
| `AVG()`           | Середнє значення    |
| `MIN()` / `MAX()` | Мінімум і максимум  |

---

**Скільки замовлень зробив кожен клієнт:**

```sql
SELECT client_id, COUNT(order_id) AS order_count
FROM orders
GROUP BY client_id;
```

**Загальна сума замовлень по кожному клієнту:**

```sql
SELECT orders.client_id,
       SUM(order_items.qty * order_items.price_uah) AS total_sum
FROM orders
JOIN order_items ON orders.order_id = order_items.order_id
GROUP BY orders.client_id;
```

---

## `HAVING`: фільтрація за агрегатами

* `WHERE` використовується до агрегації — для фільтрації рядків
* `HAVING` — після `GROUP BY`, щоб відфільтрувати групи

**Клієнти, у яких більше 3 замовлень:**

```sql
SELECT client_id, COUNT(order_id) AS order_count
FROM orders
GROUP BY client_id
HAVING COUNT(order_id) > 3;
```

**Клієнти, які витратили більше 5000 гривень:**

```sql
SELECT orders.client_id,
       SUM(order_items.qty * order_items.price_uah) AS total_spent
FROM orders
JOIN order_items ON orders.order_id = order_items.order_id
GROUP BY orders.client_id
HAVING SUM(order_items.qty * order_items.price_uah) > 5000;
```

---

## Вкладені запити (Subqueries)

Вкладений запит — це `SELECT`, який знаходиться всередині іншого запиту.

Варіанти використання:

* У `WHERE` (фільтрація за підмножиною)
* У `FROM` (як тимчасова таблиця)
* У `SELECT` (розрахунок значення)

**Знайти клієнтів, які зробили більше замовлень, ніж середнє по всіх:**

```sql
SELECT client_id, COUNT(order_id) AS order_count
FROM orders
GROUP BY client_id
HAVING COUNT(order_id) > (
    SELECT AVG(order_count) FROM (
        SELECT client_id, COUNT(order_id) AS order_count
        FROM orders
        GROUP BY client_id
    ) AS subquery
);
```

**Знайти замовлення, де сума перевищує 1000 грн:**

```sql
SELECT orders.order_id,
       SUM(order_items.qty * order_items.price_uah) AS order_total
FROM orders
JOIN order_items ON orders.order_id = order_items.order_id
GROUP BY orders.order_id
HAVING SUM(order_items.qty * order_items.price_uah) > 1000;
```

---

## Навіщо це потрібно в аналітиці?

| Питання продукту                              | Як вирішуємо                             |
| --------------------------------------------- | ---------------------------------------- |
| Які клієнти приносять найбільше виручки?      | `GROUP BY client_id + SUM(...) + HAVING` |
| Скільки замовлень у середньому робить клієнт? | `GROUP BY client_id + COUNT(order_id)`   |
| Чи є клієнти з великим середнім чеком?        | `AVG(...)`, підзапит на суму / кількість |
| Які товари продаються найчастіше?             | `GROUP BY product + SUM(quantity)`       |
